# Grouped-query attention for one decode step

In grouped-query attention (GQA), $H_q$ query heads share $H_{kv}$ key/value heads, with $G = H_q / H_{kv}$ query heads per group. The KV cache shrinks by a factor of $G$. During decoding there is one new query per sequence, attending over that sequence's cache.

Write `gqa_decode(q, k_cache, v_cache, lengths)`:

- `q`: `(B, Hq, d)`, the new token's queries.
- `k_cache`, `v_cache`: `(B, S, Hkv, d)`; `Hq` is a multiple of `Hkv`.
- `lengths`: `(B,)` int, `1 <= lengths[b] <= S`: only cache positions `< lengths[b]` are valid.
- Query head `h` uses KV head `h // G` (consecutive query heads share a KV head).

Each query head computes $\mathrm{softmax}\bigl(q k^\top / \sqrt d\bigr) v$ over the valid positions. Return `(B, Hq, d)` float32.

Do **not** copy the cache to `Hq` heads (no `jnp.repeat` of `k_cache` or `v_cache`): reshape the queries into groups instead, so the computation reads each cached key once.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def gqa_decode(q, k_cache, v_cache, lengths):
    """q (B, Hq, d), caches (B, S, Hkv, d), lengths (B,). Return (B, Hq, d)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0, 0.0], [0.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 0.0]], [[0.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 1.0]], [[3.0, 5.0]]]], "dtype": "float64"}, {"__nd__": [2], "dtype": "int64"}], "e": {"__nd__": [[[1.6604769229888916, 2.320953845977783], [2.3395230770111084, 3.679046154022217]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[1.0, 1.0], [1.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[[[0.0, 0.0]], [[5.0, 5.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 2.0]], [[9.0, 9.0]]]], "dtype": "float64"}, {"__nd__": [1], "dtype": "int64"}], "e": {"__nd__": [[[1.0, 2.0], [1.0, 2.0]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[-1.206849486366775, 0.13135791792025192, -0.23099709844912733, -0.524214950954354], [0.2913375877202091, 0.9283796450976365, -0.01076708205980992, -1.180755936768125], [0.40944321913978243, -0.4605191893082614, 0.9960779246314662, -0.33600306388197887], [0.7051338850492833, -1.1624498011773094, 1.7114252920853452, -0.8048043579065902], [-0.5743694360505074, 0.35320571112754784, 2.3407083031072706, -0.06547189650091674], [0.21592832517695004, -2.241410560281936, 0.9165219614056551, -0.30936479700256264], [0.6868991435737978, 1.7397536244113228, -1.6623563423296528, -0.3371495833115974], [1.2161881020632221, 1.4012892586441275, 0.37498554620942876, -0.9893672319639791]], [[-0.811768885288416, -0.7492212099372437, 0.21840801689847852, 0.49327418985155713], [0.17694218356679442, 0.1522189875579224, 0.3581404895311983, -0.7894507680290866], [0.030140821884032165, -2.0088910142801453, -0.8437964257099817, 0.7592845667258223], [0.4876626689472817, 1.281657598099059, -1.5986711019659905, 0.7713578422506695], [-1.0514623828599634, 0.6795356436352563, -2.026785659844316, -0.7649784054287279], [-0.3698408440821317, -1.6842890499142649, -1.049609827423383, 1.9405104124155463], [1.0224934838671147, 0.3594608999386223, -0.41938173865978956, -0.4936741592715621], [-0.4859070011418519, 2.523815031127357, 0.4802987560464868, 0.7470960184201264]]], "dtype": "float64"}, {"__nd__": [[[[-0.4077944289449384, 0.10375772827346967, 0.5809353570721465, -0.31491502990742276], [0.1876265399112365, -0.07398089218018072, -0.1161317856501904, -0.5804344668720217]], [[0.617715833816127, 0.04594788752301675, 0.9848393283895327, 0.3821960940563079], [-0.8602043565844628, 0.4051308388008776, 0.3803952298708388, 1.1623978418407863]], [[0.2827608969098076, 0.4963525988007696, 1.3994912068398828, 1.0943680176912556], [-0.13018129125406314, 1.6507034532137892, -0.2128114766814231, -0.08865329760172178]], [[1.0541091271122374, 0.0412696186637016, -1.569451780915326, 0.02131465485523585], [0.20084058970505386, 1.5590099234142891, -0.43181637960110286, -1.112942761308653]], [[1.2912392330242737, 0.33948433932618527, 1.0659850255281773, 0.07455606112363732], [0.13561032460223268, 0.16406958929585336, 0.036915727430184495, -0.593663342092719]], [[0.6702270722883372, -0.8217202849395163, -1.0792191963233322, -0.15319325378282808], [2.0083146026442997, -2.694571175601989, -1.1119230547097332, -2.1955094353420086]]], [[[0.22521902376099198, -0.59526567749702, -1.7716786019657795, 0.7589116521868712], [-1.2433980369241542, 2.2104569338352738, 0.508893017940634, 0.6317901706842922]], [[-0.3027182401046567, 0.8376111681648529, 0.27053418639325993, -1.3682062228070297], [-0.4276755721766952, -0.9473671878940376, -0.34879454673043087, -0.5818973458840798]], [[-0.3909869819109767, -0.11555739569410127, -1.0116991389381258, 0.8624360685940321], [-2.0260795490926387, -0.9818565097306271, -1.786346198392882, 0.5338562293714466]], [[-0.5798175287504064, 0.11863309230744616, -0.9531510403345197, 0.31221072667662286], [0.28922273890758043, 0.21804096525191097, 1.0990720284646889, 0.9805492292784815]], [[-0.5864001921508827, -1.8785509465779733, -1.3934999065765417, 0.6814335421315957], [-0.5430913496061703, 0.5053829883008126, -1.6950693576152591, 0.2559542729694342]], [[-0.4635976784819253, -0.4077595441381769, -0.6735616830360682, -0.5593569712116148], [0.525809288006516, 1.2342690707007518, 0.3837114679021248, 0.9894560762920174]]]], "dtype": "float64"}, {"__nd__": [[[[0.6945819428625255, -0.733563787594798, -1.0150678542117162, 0.6860255342091761], [0.9388545413449015, 2.234247948352102, 0.532479977918107, -0.43426520973009874]], [[0.002967207722791296, 1.1718088852776283, -0.14827232929667794, 0.14673933336263023], [-3.1310885079842996, -1.486629026721457, 2.4515385882024083, -0.4058735382946956]], [[-0.34732120226707486, -0.5842730494299008, 0.15814466954403578, -0.6558842409962437], [-0.9425688362253687, -0.6373478908041814, -0.3903528752682641, 1.201650425689708]], [[0.2888979633325544, 0.11347356911091089, 0.30715202154824106, 1.2380811488319807], [0.9297139036052463, 0.44266562744485377, -0.8738966743843769, -1.0293620713178329]], [[0.8040781289263513, 0.31844721766705447, 1.9835135570279434, -0.2186967764510701], [0.7240303850475159, -0.09434612646350286, -0.11220012626201824, -0.02934260325079914]], [[0.041177310653919, 0.09778105384656556, -1.283158023591092, -0.5194086800717718], [-0.1157296535433233, 1.2580420902374887, 0.6005790703033069, -0.6142039359451357]]], [[[1.1491123712663522, 0.7169977663015988, -0.852604900811494, 1.2102341885587744], [-1.991376051000914, 0.03894365463322258, 1.5149147826737706, -0.9792264809020421]], [[-0.04927459118246552, 1.5871207278856032, 0.0036349826966238278, 0.3415137428659796], [0.561022877029362, 1.924993736512289, -0.9986928208899138, -1.2953579052193067]], [[1.9176533202332324, 0.2674809671712034, -1.1835985349239622, 0.03447408593127953], [1.0502962690278788, -0.04199005240514855, 1.1325445295551775, -0.7240870976931598]], [[0.1939880474224715, -2.2373917281742846, -0.3170779393431745, -1.0703683318443005], [1.7554619042902047, 0.672562719404391, 0.9541047621109853, 0.2864583950724695]], [[-0.8433086431766404, -0.7036789495484846, -0.394311308858558, -0.46450919589635], [-0.42316746482562434, 1.8217363577240118, -0.9671719911147593, -0.7071558999966621]], [[-1.1595477120518363, 0.6348543523340293, -0.5301647664772848, -1.0376140669871654], [1.9411866982079298, 0.13396302638741264, 0.40702544495368853, 0.9533137196175755]]]], "dtype": "float64"}, {"__nd__": [6, 3], "dtype": "int64"}], "e": {"__nd__": [[[0.2988790273666382, -0.08471980690956116, -0.3037221133708954, 0.21900761127471924], [0.33485060930252075, 0.06764418631792068, 0.12066753953695297, 0.2024594098329544], [0.27049481868743896, 0.11957625299692154, 0.229332834482193, -0.01198489498347044], [0.30899903178215027, 0.1789584755897522, 0.35664626955986023, -0.04361039027571678], [-0.8849856853485107, -0.2588142454624176, 0.734614908695221, -0.13329480588436127], [-0.10218610614538193, 1.1704511642456055, 0.5972630381584167, -0.5706843137741089], [0.13205577433109283, 0.1637197881937027, -0.3587495982646942, -0.1757449507713318], [0.23632623255252838, 0.3285118341445923, -0.2597028911113739, -0.2755887508392334]], [[1.2505507469177246, 0.6908488869667053, -0.8295866847038269, 0.5251076221466064], [0.5406951308250427, 1.1795655488967896, -0.37606245279312134, 0.4368527829647064], [1.3495244979858398, 0.6047555804252625, -0.9289101958274841, 0.824124813079834], [1.3017014265060425, 0.6450536251068115, -0.8835344314575195, 0.7004277110099792], [0.5747761130332947, 0.1979081630706787, 0.9373070001602173, -0.8256398439407349], [0.9486402869224548, 0.1590127944946289, 0.9227933287620544, -0.786424458026886], [-0.18385235965251923, 0.8911976218223572, 0.2773161828517914, -1.0719287395477295], [-1.9359586238861084, 0.05258205905556679, 1.4909883737564087, -0.9786790013313293]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[0.714838596723364, -0.05371057443129949, -0.027965311465751236, 0.2629310362709747, 0.19832581880188266, -0.8907958720016279, 0.47731838548907995, -0.5667593729232702], [0.9987013833632079, 1.7295812480885397, 0.7141583365469031, 1.1468099419368352, 0.5265572597507273, -0.15169054575605206, -2.937857345968214, -2.0583939640514655], [-0.6399054337166277, -0.5870680058085262, 1.0879253754841542, -0.6951694893239504, -0.6672823930228557, -0.9439523520832559, -1.246729409529326, 0.14289250418821042], [2.2078480140252337, -0.7751165840944825, -0.12056002814682931, 0.819167737952026, -0.3568688925010957, -0.7523271876741383, 0.411892185978025, 0.014574159941890156]], [[0.011320612097857738, -1.2823633395424134, 0.0006747664744047556, -0.04044060708429217, -0.18321694709156036, 0.7133245621994327, -0.24219943494941737, -1.233329008018973], [0.1884501931684209, -1.0115478947133967, -0.8398230876391641, -0.2369597788636422, -1.8342531067962395, 0.9592778563727979, -0.6000988927781604, 0.5546976492095272], [-0.22861209138605643, 0.6169702839503617, -1.3400952515849587, 0.1257237264197605, 1.148480981405565, -1.3962399311346938, 0.9348697853604773, 0.538462471886535], [-0.2103778555970662, 1.1235658913870115, 0.18677295290358023, -1.1204794286595832, 0.5128441118535262, -0.6670716427631361, 0.28986663326827783, -0.6204940262017177]], [[-1.711689093509021, -0.5149133963874681, 0.5666412694947547, -1.5041819677834192, 1.4753178887509484, 0.09687898717035212, 0.72687105237128, 0.4781263373661347], [0.9039820899751878, -0.3634971773726635, -1.2115653159617272, -0.9656005644377826, 0.42108327140704777, -0.5931483507340646, 0.36292898195877044, 0.07891590701834626], [1.0236155122695905, -0.7304812369205289, -1.8249241002889214, 1.0286981353701596, 0.08362920048602512, -1.8635507271723453, -0.4131761433926976, 2.192586806209013], [-1.4364546328613788, 1.1169984300157294, -0.28781805979357905, -1.7892016194936127, 0.026173400643632567, 0.2289529542484353, 1.0628018884801291, -0.7040278254923062]]], "dtype": "float64"}, {"__nd__": [[[[1.5028993903571684, 0.3370027171073551, -1.6475121787252105, 0.33528213948914265, -0.9976490789081759, -0.5921152591245342, -0.11181872583607358, 0.42730292571907713], [0.8510381928040314, -0.4667986959856318, 1.1731022145229255, -0.45928505432213024, -0.25614214895369747, -0.052671402413811005, -1.4040945207016924, -2.0287856996055798], [0.7902499124421881, 0.015879119201917362, 1.0904724653786102, 1.1889161803604156, -1.5185474664444467, 0.7223678766089855, 0.09699754729347433, -1.840423754968993], [0.5824443482763569, 1.3036207263255974, 1.4751983134041944, 1.0822554472083623, 0.8132286494302818, -1.3047483640518587, 0.9727815732785331, 0.15469909187054015]], [[-1.4372223718130142, 3.665220599679864, 0.1338035528798171, 0.698086014994682, 0.037385293812398315, 0.5552076503923911, -0.8084048133900809, -0.12625984747763422], [-0.36232123066336386, -0.3396845720061046, -0.7816977450636979, 2.920640129920896, -0.6586251024376464, 0.20724931095003685, 1.9100905251282834, -0.8752566561242383], [2.3058646446193407, -1.0440638998828604, 0.32973152583795723, -0.10431224970275196, -0.9464412851408904, -0.2130770542766429, -0.8968750902075029, -0.4057759475482125], [-0.43989445667088106, -0.384249450836254, -1.6219919081707594, -0.26109030638308495, -0.34386438768069005, 0.9034199140308352, -0.12307314199658409, -0.22564952426228505]], [[0.5180861784081227, -1.2260047092549393, -0.5516598892572494, 0.9299717386346631, -0.2871976389268331, -0.20918624532501143, 0.3518699626691965, 0.020486674814901306], [-0.8306064149194723, -0.3756440175545151, 0.6410242870330654, -0.09452526543413459, -1.0436505993489076, -1.0105872701172045, 0.13045267378985914, -0.08066021423596936], [-0.25799554186644474, 0.9370196451422378, -1.9495380557793518, -1.3471317701689136, 1.6957881261713084, -1.6039210159033195, -0.47400607076540746, -0.6056635584605065], [-0.3299530446262677, 0.722638035679525, 0.21326788383692058, -0.668315760581685, 0.3243195600365407, 0.7113437256457175, -0.535906974629909, 0.22908719929743698]], [[-0.753532828124808, 0.6549037541479027, 0.343798437437948, 1.062147146081806, -1.864132570455029, -0.7093815745566123, 1.0957478532564, 0.3596704339462992], [-0.9163650276367808, -0.5328176816985714, 0.46348504241155636, -0.26669604258211943, 0.8267997645849511, -0.22685626720718896, 0.8766811950643904, 0.7701178396379735], [0.05067039231235952, 0.29910270131361166, 0.4440165108498007, 0.17586383083307608, -1.0924972745739057, -0.6225946060785329, 1.0430306605498116, -3.241294627082675], [-0.38411639940088105, -0.16934273743828024, 0.18812846878623118, 1.6373763697058703, 0.8142582005174348, -0.9054956883533438, 0.24746742828807405, 0.35845079017036896]], [[-1.373144598883435, -0.19987501484502526, 0.1664092863304915, 0.39862096269056974, -0.9183779713909576, 0.2610723440036502, -0.49012847553625377, -0.3278507125866638], [-0.9335664536021362, -2.167953409229565, 0.04235167150924798, -0.41047202747111117, -0.7369180779408115, -1.5081354255110606, 0.8468264147010509, 0.3967706936827645], [-0.13875687756582947, 0.9111444318612297, -2.1480532950820956, 2.0647382381855226, -0.43439346684315616, -0.21659752808231922, -1.0385705967520704, 0.15967590872640192], [-0.925456109700561, -0.6982707405942034, -0.5472352738083102, -0.8500460800585241, -0.9549559001583271, 0.6862761656818103, -0.8711041716062995, -0.7254009259856625]]], [[[-0.008394994883585813, 0.3573346449400458, -0.015899357826845838, -1.4652904327574814, 0.6101713514283496, 0.3841236118380068, 0.7280521302421599, -0.5496472885383641], [0.7909996031667067, 0.971619427979421, -0.8056053352466264, -1.5504184908004202, 1.2491942448472142, 0.12527017589312076, 1.602035639642995, -0.07757181200604506], [-1.7015465489085178, -0.735199654137974, -0.6132395370275878, -0.26593747674937535, 0.050577829462235456, -0.9673688008799447, -0.7575124555030137, -0.12499088993314365], [-0.6338379902797755, 0.17129665515441023, -0.9029227048163646, 2.0924909341119773, 1.36281522162768, 0.20766138713557733, -1.0093236369844805, -0.08914839170819186]], [[0.16758366743322825, -1.1920247956365375, 0.6464242823401093, -0.5530357403485883, -1.0864240700349348, -0.9708652544225111, -0.3491143278827687, 1.3963946714229196], [-0.7253162562702054, 0.9140049064896, 1.0546991208920178, -0.7508870468454527, 0.648924440841179, 0.49866881635330307, -0.2740979888795814, 0.6959944784071888], [-0.5314425838637804, -0.7223220292542405, -1.6537938705277972, -0.4256573040319171, -0.3611173622429606, -0.052124915536467635, 1.047539333505305, -1.8130343329784093], [0.11950755465692293, -1.0016585363020425, -1.7985504079754189, 1.6378426397265857, -1.4028026168895593, 0.6030507016762565, -0.6601999045542117, 0.5766331055663902]], [[-0.2412391307996041, 1.4023913919495261, -1.0117112970725815, -0.5293018378135864, -0.134399725122544, 1.716394363791443, -0.3422444094259499, 0.2702644977004021], [-0.7158811672919474, 1.1570253894862987, 0.6024887012395647, 2.305393273318372, 1.4050426116300123, 1.1105407274952592, -0.018615463902234005, 1.1863961430543288], [1.3931189132794322, -1.017329466749298, 0.8508095743428115, -0.8945078084070515, 0.7540253462776259, -0.5215877989950766, -1.2689205102477965, -0.5022047847497658], [-2.6536160156927533, 0.6615560475268765, 1.685648584360098, 1.3718510972445277, 2.275774043431129, 0.7137866271443055, 0.25753245047317846, -0.8413435634008238]], [[1.0573723508358956, -0.041154279218869064, -0.4531718764363845, -1.4827145012495206, -0.9917150824661601, 0.7127625703222333, -0.5514732675209407, -0.30015944144427925], [0.4051046050456585, 1.3151801072413978, 0.8164255635855788, -1.5938475619010517, 2.1140064665887066, 0.25544080243310757, 1.1715730473168446, 0.8124516754058155], [-0.9235657101584799, 0.7981505678124544, -2.3685029302561342, -0.5858829732566452, 0.3496593860337964, -1.330391390395944, 1.3534751443915067, -1.5804199867372226], [-1.5677758892826694, -2.7275844675968637, -0.14151472171762022, -1.649792781978404, -0.010680364242007589, -2.3090209214099318, -1.304203272941545, -0.07858728617669189]], [[1.471219214142776, 0.9102574742444312, -0.5799317208334007, -0.2097599479470598, 0.15277635136575862, 0.0783575444933068, 0.4593968229740487, -1.3963077338319976], [0.0013369540499303491, -0.802161122990898, -1.332263873217523, -0.09846665273809303, 0.6121631932864822, 0.34630443312110315, -2.0230549796185264, -0.22827216939481698], [1.5162641407209996, 0.7101380089701433, 0.3683612598886024, -0.31621503991661404, -0.5055025002945845, 1.3173206588470743, 0.41210877226612563, 1.4626527993123795], [0.02916646468002188, -0.648667909123304, -0.6351243400101774, 1.5823657617290097, -2.6714358528564843, -0.4719736079462704, 0.6156831257103655, -0.08162978645721665]]], [[[-0.6398488153511731, 0.05128141755680987, 0.9922237403976395, 0.5763534341588831, 1.0959435439049638, 1.0698629318726984, 0.36459983998866835, -0.8596317515452243], [-0.042864639327549206, -1.247561454156686, 0.08358050876336792, 1.3639174891497103, -1.256107193235695, -0.3177584375603594, -1.5166580090843278, -0.8854714142398271], [1.0321012044617053, 0.1352657141057475, 1.1212030609723336, -2.454238921570653, 0.8380654294839847, 1.6788964009036007, 1.899284154594371, -0.4926352119473475], [-1.092195620846344, 0.5553685280483764, 0.2973748478923952, -0.4522040462220015, 0.607878033242334, 1.5446327512883056, 0.6352109870279364, 0.416012824437035]], [[0.4688659558067621, 0.6353646008845977, 1.3079939379727483, -1.5783313026361916, -1.0686398352964226, -0.4803131621188658, -0.8142258465884077, -1.1681336452392348], [1.0703169351353572, 0.7770316938543536, 1.5588161439610162, -0.6816748838743136, 1.298627038198189, -0.6297750245160741, -0.10646344830355292, -0.3955318545367217], [-1.4276932605852202, 0.5485401945310742, -0.6858044416972126, -0.0631058910483341, -0.42116510334439927, -0.8336304406735525, -1.667967966259879, 0.007034679258578392], [0.12271788946770827, -2.512215569080574, 0.36135183993506587, -2.2619323905436595, -0.9874216619577914, 0.881980575521095, -0.5997365334414131, -0.9059038850754791]], [[1.1689928596889068, -0.42101279240842376, -2.2476627233518536, 0.4354632195306479, 0.6061848007492022, -0.2225270081338467, 0.12925647860094694, 0.27843433573932047], [-1.1746494647341064, 1.0944982721964558, 1.2324971528228499, -0.36554777057412713, -1.0529821789627372, 2.427537271951647, -1.8690744874966512, 1.2789389940984455], [0.223223911392561, 1.0463047746161822, -0.6962806462039308, -0.016468189270957143, 0.294996249529003, 1.6017749123327283, -0.3557391170439729, 0.9041426760799645], [-0.270734870752942, -0.5014252204325154, -0.5793185979364873, -0.482214351367032, -0.7645504094894569, 1.1696226328512054, 0.1803513594503619, 0.984476182798598]], [[0.6263266702993455, -0.9285567447414504, -1.8082975295217998, 1.308688805527772, 0.5456826259710599, 0.121554288031101, -2.1366712696973518, -0.7850887951640828], [0.5956786513601002, -0.016196197242605393, 0.16636087744287364, -0.730489725743209, -0.4421567971652505, -0.8811434811430906, 1.2869929681752348, 0.6053303436374952], [0.2811444258987521, -0.14281937365476802, 0.03608282697114945, 1.5677695514065708, -0.5683103314642313, -0.7423769948775594, -0.42451142255341157, -0.5305360342585954], [-0.9238620846542258, 0.39773958005024745, 0.11853629750506038, -1.0405709605059152, 1.5010782267076797, 0.8931965459790268, 0.2633110931313546, -0.6627829720291618]], [[0.18411205271221404, 1.524404445605022, 0.6345356702407298, 0.29643679805007006, -1.4615286329758777, -1.0649359855786913, 0.35953404963074426, -0.8870173746267538], [0.2916447339571684, -1.2569564844338244, -0.6389670831734483, 1.38753727052099, 0.5818325374554847, -0.08907919515566581, -1.1565718312930096, -1.0838700273560022], [0.15221634280014457, 0.8561698701551036, -0.5935482330409114, -1.3692535869582856, 0.5129122046456165, -0.08665135656604266, -0.7314018041778043, -0.018290273434501108], [-0.6979637553752321, -0.5853466605510159, -0.23789058494898765, 0.014170243248536336, -0.7034440993324933, 1.5395656147592252, 0.8403798574169264, 1.1878722451495642]]]], "dtype": "float64"}, {"__nd__": [[[[0.44744106845880116, -0.7311611356391429, 0.5264094006836312, -0.5304153125679274, -0.0667189857414196, 0.2702774725403144, -1.5648169922741333, -0.45747871832154097], [0.16941627813026613, -0.9216198472189752, -2.3015203654685026, -0.1651025695815223, -0.2291946505534875, 1.4820043940380379, -0.18446520078175083, -0.25335989765755074], [0.5709823855002915, -0.5729450698129954, -0.21754453921303363, -0.48999731605001734, 0.17384704822281832, 0.15032369348132757, 0.09669391661819264, -0.32097478727560297], [1.295546892632663, 0.6461835772526053, -1.7495151226396757, 0.42919241280721254, 0.16935708526045185, -0.19841990151199998, -1.1999194231215515, 0.11819739406878145]], [[-0.7247405933114454, -1.0563875808120815, 0.7158244976467514, 0.5635000809073036, 0.2107788654401526, 0.32585921398436835, 0.579126688186587, 0.6933608414097913], [1.4134343920121608, 0.36242732816494233, 1.7547197761557969, 0.47392808558728877, 0.5570314906472257, -1.6925888405955407, -0.9027040770114352, 0.44700794244312664], [0.17690783494757356, 0.466371057024896, 0.6318544044042007, 0.0031822392108942915, 1.235114354622566, 0.4698051957618794, -0.34557544739490587, 0.6806767894880243], [1.152638575422535, 0.6357115900047351, -1.8680305454128587, 0.886792343446659, -0.38759278428332633, -0.41378665312851426, -0.4929723722981639, -0.3477933983451653]], [[0.636168572582297, 0.2641441419790647, 0.17056973323944288, 1.1946477022926574, -0.5756196850844896, 0.3211844157728021, -0.5911843048778828, -1.215786427484023], [0.8716470196508032, -0.9298758420594556, 0.8865865854914915, 0.34020509820097133, 1.0751086254186637, 0.47279998864984324, -0.2813919148277668, -1.002889279910585], [-0.20057756341952285, 0.685698498220405, 0.5439532029997243, 1.660671636867844, -0.7240084281930196, -0.495149392580015, -0.7108212003120438, -0.2851747539445911], [2.076439912929648, -0.5190934875284846, -0.4290959553627889, 0.14464421526945856, -1.0487300967618416, 0.49657237889753664, 0.6499584274842753, 0.2111895997776147]], [[-0.3597945724907076, -0.08448970507779584, -0.19932016530660232, 1.3881161807314197, -0.30860752483752535, -0.4185309347643216, 0.6070136562455427, 0.5882425868719581], [1.7029223657290495, 0.9833152670557128, 1.8731017558277878, 1.1370826947165686, 0.7960477768577796, -0.011331434900278017, -0.8861619674864392, 1.8702920890817714], [-0.5174220788585813, 1.1515507076336446, 0.24985342694440646, 1.38255402725303, -0.5459982730306862, 0.6468275301345194, 0.545022456054212, 0.007865133435339447], [1.4035050125591262, 0.11983490432832707, 0.969941552983357, 0.11969698195712423, 0.7070519517780726, 0.9426691899387011, 0.17073967171628215, -0.028979476690635636]], [[1.6510520692820054, -1.906351708375482, 0.9893638462669926, 0.9274923692028383, 2.4835572362045344, -0.6990413855011045, 0.9095200264550882, -1.9335878771942299], [0.6031451307174822, -1.186975777310731, 0.6916291224671035, -0.747295004038468, 2.0322847548342833, 0.17678032512231473, 0.3460628931800258, -0.16760588796507056], [-0.8047326216931496, -1.0618826866470894, 0.6906196117308537, 1.6105991376243685, -1.0030928457363664, 0.1449303469089825, 0.0977203059945371, -0.8858695005119889], [0.6449484426530646, 1.7760936289811995, 0.9350452141314839, 0.20525383465960192, 0.48377209785953, -0.9037792552022327, 1.4230850436037188, 0.08640437866146228]]], [[[-0.30738024121949165, 1.5805131328289528, -0.804634225793019, 0.28442778655990647, 0.10065260375399926, 0.5632546867086051, 0.9834067839538423, -0.17508794795133484], [1.805154930814107, 1.5579512052659241, 0.12234739327283631, 1.6373626470716298, 0.90211260512181, -0.3148877313176983, 0.03557326507419324, 2.666586332179174], [1.6380926387151435, 1.034903850198337, 0.8111982060873327, -1.541355769085226, -0.7715708907336937, 0.7148154941706951, 0.5393435190403552, -3.719538847440336], [-0.3958545634532231, -2.304436229901386, -1.2693839106975415, -1.3764600900745265, 0.05988760518886403, -1.1279533139480935, 0.05935101609364929, 0.04838553136539388]], [[-0.02170543426785061, 1.5120336930447946, 0.6164122832097624, -0.4761095376361906, -1.2725272730275305, -0.48343037522480314, 1.194120416163151, -0.6187706135554014], [0.9806289893014526, -0.34214075150707657, 1.9076498800987083, -0.5783863193593086, 1.151385959918493, -1.0969466846063542, 0.6153128932754692, 1.320284437034456], [0.47570558810965347, -0.3766008940755841, 0.7374238324243308, 1.5882961677324583, 0.14441972260699815, -0.8041190847114233, -0.9949893327742849, -0.843195450528464], [-1.2977976409435792, -0.9027031102760122, 1.6998550043349918, -0.7191672377346726, 0.7091062306395971, 0.5105990905272785, 1.357154714193341, 0.0782658392005912]], [[-1.1068443842731344, 2.3339441300237955, -0.288033804839845, 1.8566272298981092, 0.19209281650165538, 1.2143020618947615, -0.4705782682013032, -0.4816122422330042], [-1.040383476751948, 1.6534983495032187, -1.1223818733162652, 0.8417981846003985, 0.13912493223232353, -0.5933303920331897, 0.5721640206304575, -1.1880551127498742], [1.039949907604428, 0.36569302419280997, 0.051141926861895934, 0.9812029643942962, -0.165299754845716, 1.8407219511898012, 0.4869760785906086, -0.14707391042491014], [-0.7973870868895945, -0.2522748234061293, -2.454989260778281, 0.7383749971985719, 0.07724841677194645, 0.6274009601834982, 1.05771686542309, -1.8065763120399563]], [[0.08470367390494542, -0.696800574216673, -1.3601401598589882, 0.32278535316467805, 0.1585846413451847, -0.7605193587524491, 1.7279916261440418, 0.5780168118310197], [1.8676768027453532, -0.6426682063433002, 0.5611368301412031, 1.4894349709235728, -1.217225371254151, 0.3548274459779783, -1.143094039827514, -0.38842370720468117], [-1.6649721108372981, 0.7523907207561292, -0.028069001218765377, 0.8745610130455201, -0.025850293236173195, -0.2370219089804091, 0.20848301204448122, -0.5644755289449497], [1.072237720164936, 1.60458307866277, 1.4966816758819343, -0.014634175836338846, 0.9753894250443895, -0.9549679999429503, -0.030821100923536734, -0.5419393503783811]], [[-0.4835522387326852, 1.389634466207709, 0.68498953208916, 1.1889039593044473, -1.3191583827628817, -0.5803685637498903, -1.4488153173786908, -0.546182879595058], [1.75498142303113, 1.0594609077849413, -0.031538867927618695, 1.4737299210245447, 0.42219244299499087, -1.7252770945443325, -0.5969883988497544, 0.5618624041210668], [1.0495273989200709, -0.7332910591450194, 2.1309167298550333, -0.47681682527015334, 0.7491809109937241, -0.5982153430411669, 0.2724284934036754, 1.0197457844602886], [-0.6631917024445897, -0.6237673258368202, 1.1037844776077959, 0.8327647495568273, -0.7819313807315668, -0.2709399928125047, 0.48926487151414405, -1.5745499041528153]]], [[[1.3593915514516934, 0.6744241342632263, 0.7165871854251641, 1.0399514126791225, 2.254803371123709, 0.541322458951591, -0.3851841646685046, 0.4839078351160577], [0.275904215998541, -1.5118400028532124, -1.0865720196382014, -1.4342839878728304, 0.06489122231862483, -0.853793330481854, 1.2749775636832419, -0.3591175361334452], [-0.566643565791218, 1.142003037470463, 1.7137109191723632, 0.10062205419900265, 0.18922089132477515, 0.7220793113790451, 0.4468821301580654, -0.674185952365684], [-0.4964673794425423, -0.6710320032600803, -1.3400356421205974, 0.3295387921453466, 0.3363114008939561, -0.39830334449068167, 0.8171760682246165, 1.0596151587868254]], [[0.38495864435513794, 0.27678889230244696, 1.7543778728219834, -0.7779606517706388, 0.4660088058925353, -0.22445107922408405, -0.5461289992557994, 0.8616669346515794], [-0.7479525702352455, -0.2967180317815369, -0.21094737950237352, -0.2874379203348481, 1.8382108810411373, -0.17697605994634255, 0.336147077036934, -1.5559843522696675], [-1.282955251312336, -0.07543451979557894, -2.2501727737640493, 0.19016226362639907, -1.8161723100190843, -0.14561105359387808, 0.44851585020869883, -1.000500387950594], [-0.2792667465096941, -3.018465876273061, -1.4254563854828861, 0.7556686136716442, -1.2162440969375476, 0.31558279062189165, 0.9542779821650175, 0.4786380467522943]], [[-0.7164793249754183, -0.15415529645350265, 0.027009622047502534, 1.4843873456419607, 0.4303238035247679, -0.6487407776767339, 0.07434879098708067, -1.9253234218235333], [0.7231478584677499, -0.6107472543989293, 0.9707242283955774, 0.43157125473792407, -0.2196706881142593, 0.3219263804045532, -1.3677937161433948, 0.3479477730054119], [0.010986688750433142, -0.3530916163459022, -0.6376174875041832, 0.320849466099417, -0.35620411893950765, -0.06271730927120028, -1.4157779149530993, -0.6770658698392747], [0.6587962233583687, 0.39223292891821426, -0.5305303644095161, 1.5470781847097834, -0.24680122621028927, 1.6485864225271911, -0.7020434832340702, -0.4787689208352994]], [[-0.4036100726342376, -1.0937390770882562, 0.46513661288332964, -1.3748343996021206, -0.9628093525159968, -1.6347010064619407, -1.0627476934202327, -1.3471877125530214], [-0.9196433707774149, 2.1467773850327627, -0.3461715031409921, 0.02688104452801859, -0.18318465438621934, 0.28598113253847396, 0.5517812832620022, -0.5493532798231658], [0.727660435468642, -0.8200512086541188, 0.7948013579974595, -1.591918313288158, 0.028411465822849626, -0.5867271973901051, -0.7630354274461868, -0.11233865130157489], [0.30470814530842527, 0.00014115103874947493, 0.32897317974232626, 1.1023950920212946, -0.4376819219306124, -0.5543749300157824, 0.44832657726552766, -1.0754642002547041]], [[-0.6184994475588428, -0.29873074786171755, 0.524984830943756, -1.5584670047093963, -0.008009048329836135, -0.7884385028650794, 2.029202450441709, 0.1261731239903101], [0.4760750716624026, 2.0909370968942973, -0.7010662081183949, -1.4360499326816878, 0.08016431346544373, 1.2723430546326073, -0.12256909720244102, -0.428513507778411], [0.43682826755932613, 0.7931011511502501, 0.46932272838230454, 1.6445595355288463, 1.2949474045634346, 0.03247767880265253, -0.8936125694425143, -0.8982575201883479], [0.7583138116556108, 0.31798119692144583, -1.205373299421143, -0.777027462258797, -0.3488925379535019, -1.744021207368952, -1.8699528412023874, -0.5577692230231032]]]], "dtype": "float64"}, {"__nd__": [1, 5, 2], "dtype": "int64"}], "e": {"__nd__": [[[0.44744107127189636, -0.7311611175537109, 0.526409387588501, -0.5304152965545654, -0.06671898812055588, 0.2702774703502655, -1.564816951751709, -0.45747873187065125], [0.16941627860069275, -0.9216198325157166, -2.301520347595215, -0.16510257124900818, -0.22919465601444244, 1.4820044040679932, -0.1844651997089386, -0.2533598840236664], [0.5709823966026306, -0.5729450583457947, -0.2175445407629013, -0.48999732732772827, 0.1738470494747162, 0.15032368898391724, 0.09669391810894012, -0.32097479701042175], [1.2955468893051147, 0.6461835503578186, -1.749515175819397, 0.4291924238204956, 0.1693570911884308, -0.19841989874839783, -1.1999194622039795, 0.11819739639759064]], [[-0.29978910088539124, 0.9682539701461792, -0.35748282074928284, 0.5980523824691772, -0.3849042057991028, -0.13894948363304138, 0.499099999666214, -0.14507246017456055], [1.3993504047393799, 0.8927869200706482, 0.185513436794281, 1.140068769454956, 0.4805028438568115, -1.3071712255477905, -0.2772692143917084, 0.6799244284629822], [-0.8372275233268738, 0.5789973139762878, 0.24494753777980804, 0.6367234587669373, -0.06495772302150726, -0.14014355838298798, 0.11421168595552444, -0.880706250667572], [-0.19554267823696136, -0.043090786784887314, -0.6355997920036316, 0.15926505625247955, 0.3039093315601349, -0.16637521982192993, 0.5622374415397644, -1.0652554035186768]], [[1.1170214414596558, 0.5755205750465393, 0.9747161269187927, 0.5877833366394043, 1.809877634048462, 0.3508521318435669, -0.42521587014198303, 0.5778676271438599], [-0.4310566782951355, -0.6728127598762512, -0.4819636642932892, -0.6424004435539246, 1.2893472909927368, -0.386459082365036, 0.6267263293266296, -1.185539722442627], [-1.263248085975647, -0.041940487921237946, -2.1411187648773193, 0.1876988261938095, -1.7610000371932983, -0.1217392235994339, 0.4484708905220032, -0.9915228486061096], [-0.43339821696281433, -1.3526631593704224, -1.3648395538330078, 0.4532753527164459, -0.11450859904289246, -0.19101017713546753, 0.8569867610931396, 0.8909151554107666]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[1.3156978733517752, 0.5931452362938732, -0.8512938663886894, -0.43508407975607843], [-1.3526002967227793, 1.4579036367247722, -0.11337170154761453, 0.7341836047654318], [-1.3825194904437061, -0.3930091585969842, 1.5145242428432393, 1.5989396740141004], [0.4237727368005683, 1.3211338261770897, 0.48026580254933376, -0.45590864748823917], [0.47258945242657635, 1.1921195356373344, 1.0964496547673654, -0.4366594973220239], [1.0967579036364459, 0.4000330328971295, -0.9362662199748969, 0.26771553929880404]]], "dtype": "float64"}, {"__nd__": [[[[-0.1200892247916342, 0.17379094758510513, 1.1982241249124737, -1.3579257438596972]], [[1.0179542384541933, 0.9149049463848478, -0.2792812103598713, 0.4526977785229335]], [[-1.9328211412191414, -0.5985564419937763, 0.02942278135453453, 0.6796752070858972]], [[-0.24767466942500332, 0.3648741064985899, -0.19908815328717056, -0.3251372838337291]], [[-1.4528462329819698, 0.5428650351519217, 1.2117709575826148, 0.28673370798850484]], [[2.292623092717192, 2.505806177213157, -1.21453352167408, -0.3408482602133749]], [[-1.261488778036441, 0.23422866341594056, 0.7520068501461936, 1.5003739242431493]]]], "dtype": "float64"}, {"__nd__": [[[[-1.4385552096842282, 1.9473287293008994, 1.0808331194260803, 0.7504995306389597]], [[-0.8540975455783887, 1.322798008317616, -2.0737795743407283, -0.9144097317466219]], [[0.9350511407318487, -0.8431193027551973, -1.933870831139717, -0.21214714354551342]], [[-0.5839410460723623, 0.8157358773109195, -0.5642197149850722, 0.561995225855065]], [[-0.11397174011299238, -0.12948985924486672, 0.6720761908872488, -0.7422529231403875]], [[-0.013153059752586998, -1.578608779795468, 0.5546160943722347, 0.05409938796033706]], [[1.636460013910747, -0.10196226204206614, 0.48880976292749834, -0.5990109117774611]]]], "dtype": "float64"}, {"__nd__": [7], "dtype": "int64"}], "e": {"__nd__": [[[-0.16436424851417542, -0.9579280018806458, 0.18871302902698517, -0.028159964829683304], [0.43740081787109375, -0.09253375977277756, -0.16316930949687958, -0.37896573543548584], [0.8440400958061218, -0.18950460851192474, -0.10615010559558868, -0.47825416922569275], [-0.2662270665168762, -0.2765716016292572, 0.12423434108495712, -0.04887223243713379], [-0.3480413854122162, 0.09869685769081116, 0.16224971413612366, -0.04348010942339897], [-0.1698678731918335, -0.7237908840179443, -0.00812461320310831, -0.09782227873802185]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[2.0315578504124856, -7.185985507691061, 5.081366937025191, -0.9386041814800952, 0.14242903935593496, -1.705615674826083, -5.960363878513601, -0.8704131746711015, 8.126971488733414, 2.6871406881604063, 2.9905996106738355, -4.1649149715995675, -2.9394892982633367, -4.174418590701719, -5.844137906344619, -0.4448111947135661], [-2.9043814868267726, -2.179562532563559, 1.4193771273373152, 0.8923338921118057, -0.5996912683140766, -3.843987543750479, 4.913974615555954, -3.0942184091197045, 7.647670271585888, -1.9251190147833381, -6.296926768094833, -5.836525006007753, -4.633448549855619, -0.7992877792317863, -1.1208826064968527, -2.004873377029629], [-1.4795265368110755, -8.8658631896014, -4.139249556084897, -0.9529386263207099, 6.712144536703089, -0.8676588053735742, 2.6705731262920938, -0.25936518562805344, -0.1989343527558931, -0.7792675536074559, 5.464122234916149, 2.0813477312824826, 4.756490878739427, -3.0368789554623232, -5.2076671272646555, -3.2553499414038165], [3.684124654396107, -3.43224491049523, -0.5953949153288449, 2.18988091910433, 3.7873077037363156, 0.12854146065238847, 0.33588507477079027, 3.2160617554104416, 7.571695153938138, -4.846679852104842, -3.678436707972863, 2.132734070989218, -3.648609605843487, -8.190702572432196, 1.7226046789167675, 3.7461095089252847]], [[-1.6892817084264966, -4.130094030035823, -1.4715520858079658, 0.7496592706835963, 7.252045862140279, 4.173766677084479, 3.231925990841781, -3.2140691080249093, 3.658267107574427, -2.1593974556316593, -3.9213441368425648, -2.692761035682955, 0.0858448352582431, -0.7019462276972788, 4.253919886683844, 2.2245006284845674], [-2.8391350030616254, 5.245522067083648, -3.21301987627608, 1.1575199006752257, -4.4111210292098475, 0.10183920468368128, -1.5182994391141738, 7.309947500325716, -2.800714631440889, 0.4692978390559949, 0.901499570468565, -7.270458482928754, 6.1175094436592765, -1.2073397877438456, 1.0460729090579624, 6.835409082906015], [3.3782817415250044, 0.7199308733124211, -0.8623906643460005, -3.923125225745649, -2.4126235233108013, -2.3256791763363087, -11.49027815533997, 2.97348493782362, 5.821876943554872, 2.3218391238512974, -7.259169787714571, -1.0935857603301775, -0.3287649364849135, 5.128250675567962, 3.5020847857565975, 4.726262413874408], [2.095351325665428, -2.506657422459094, 0.04487492258847893, -4.966573406391436, 5.481759648718684, 5.1912269270567055, -3.4670122647264905, -5.409060172011868, -1.6777493036149624, -2.4104254130747873, -1.1388532771058633, 6.773159076971656, 0.39913966503561954, -3.204489988939558, -2.97703870571907, -1.4253920764133865]]], "dtype": "float64"}, {"__nd__": [[[[3.4658719868487586, 0.271259316352833, -3.8870126890480208, -2.9374781453270504, -0.8849646916966335, 3.6664872506987862, -4.5322825807098575, 6.679113336392244, -0.9340075155344223, 5.567507952254469, 5.478772188642023, 1.3918528122272589, 2.4865851184462375, 3.7670209150724703, -2.220719994321256, 3.7414290147131246], [-6.239540649154073, 7.840212053072545, 0.10104635682936067, 1.5493265988987224, -0.48443796874989337, -3.9111771905614345, -6.053543036011032, -3.1284281415875697, 1.764262091464565, -3.124047160685524, -3.6257307885553933, 3.6977445042949046, -0.9989197050822167, 2.744598589427591, 0.6712321206841099, -4.468561647195309]], [[-0.03109837077310735, -0.28692130160047846, 1.5785509083639342, 1.3908402249721579, -4.633606135327246, -0.911646954981576, -1.8844081281401874, 4.131460972702909, -0.35738996246555743, 3.53258901019716, -5.162279110082112, 2.1714730987576316, -0.031805422450943505, -4.075507936202913, -5.259196210124046, 1.8006174373187935], [-1.9999604894827239, -2.196688544860228, -1.4275697925765736, -2.6442645466316907, -7.1342588810253265, 1.4590036582119958, -6.475436485816386, -4.821150614578617, -2.954502075656986, -2.9302873152270266, -7.3441926668532735, -2.9312329808649844, 6.47484879133668, -1.280523851822144, -3.5976334232552554, -5.595869196516118]], [[-3.910420419557619, 1.5623572617580213, -3.5037015336271957, 0.4914628189445368, -2.686674607246883, 4.372174872306165, -0.2884988034004606, 2.1053034839908826, 0.8805702741520777, -6.9240770891765075, -2.2675023128199414, 0.6009164291609942, 4.651782174334566, -2.8748926020347776, 4.066436243921883, 0.10409470138118723], [1.380862812364755, 3.5022749220429175, 1.9063836814643156, 6.275329353645322, 9.329618041241924, -0.8273207405961989, -1.392837109185875, 2.274471953882976, 0.5613911163907856, 1.2794956262253332, 8.040048595279641, 4.332486529307355, -1.8346529699800238, -0.09860256758643887, 3.262617515300165, 3.5313076551027063]], [[-3.0513775750181176, 6.1737417472129685, 2.304593958254483, 1.029469680762241, -3.301023039604829, -6.494841819886513, 1.6894837757311734, 2.579304707648203, 1.7787100877010396, -4.461293434980906, -2.043606692547096, -4.723445468187117, 5.157386497841074, 0.8812040460881282, 3.7812740863053143, -0.3742077752822158], [-2.4535714860448032, -4.282659605776463, 5.023173481575066, 1.6766307642008564, -1.2683441507267796, 0.8471978352654259, -1.3959822631713297, -1.761165582425415, -1.0994988966477712, 0.023706046458231333, -0.2223906149146695, 1.2627243933420007, -1.9404664093213233, 1.4396011192713287, -1.9718445069396469, -4.115546459452489]], [[-1.1369680544936143, -0.9986825615352541, 5.258428907153187, 2.5770407804457505, 0.5225723304035774, 2.9291735090838014, 3.211344005093473, -1.8515052253084574, -2.155499958301239, 4.667179960326904, 4.2848722280184175, 1.9598603304434667, 0.36550115098617897, -1.2029917310346412, -5.2420236693666515, 2.839009064989442], [1.6099525797519973, -3.2806343322576756, 7.047996581709239, -5.575502199693467, 1.4767185227298107, -4.138667942818092, -2.0067570004980277, -3.312476464688755, -5.314866856626685, -4.862804851214662, 1.9707513805457586, -0.8380144992308458, 1.1662422683009837, -2.101867618088513, 8.299465219555108, 10.01929571999452]], [[0.5421932284665775, 5.200859749521512, -4.976156867871279, 0.2230227797973234, -6.1029515705011255, 1.3263182016197959, 0.6051000454418006, -8.336464884287308, -3.8411609439059933, 1.6798243513120004, 4.161067477531911, 4.029677391786368, 1.4395048238670691, -5.147270820011201, -0.6949962387082945, -5.348205038171308], [1.735688725596691, -0.3457474387403622, -6.392504345255213, -3.3522031770070506, 2.2159703188796227, -4.860001653800966, 2.4725264245760092, 4.6335175145260825, -3.1146974239901635, -6.978252638729868, -6.295614239033266, 0.6009215629135918, -5.188223490998578, -2.233333178404646, 3.508153311419163, 2.737339161017035]], [[-1.731412190002555, -0.9793179255273459, -2.357105349523899, 1.4375456337242605, -6.372328587106006, 8.304375852721055, 3.445820930440614, 2.5755448704422164, 4.942378851797868, -0.41406698115100793, 6.1015249085283285, -0.19960485578191492, 3.7078695775158934, -3.9797518889910135, -0.45210960347935747, -3.1718524615144394], [0.6061322880213325, 3.905375815549343, 4.390217899173503, 7.903591389663006, -3.9943843336443163, -1.9072792462319523, -3.760715041982029, 4.458103903429783, -1.7257652816371098, 2.7395986342976464, -2.197565545191737, -9.221931044438161, 3.1344949911786477, -5.649662347755961, -1.9022632336114493, -4.482077571059215]], [[0.14919342842505845, 0.19877353560461572, -5.518892390935097, 2.106860603645549, 1.5624324182595428, -0.5973367659831653, -3.8222225748094187, 3.454517974916189, -3.447963230607857, -2.1141304417839386, 0.35497062575710264, 3.060068743804109, 6.348777242524087, -7.452060806364335, 4.115914078340118, 3.3247203220461303], [5.322980688097516, -0.8192413346767137, 3.8660471828010574, 6.038131915290291, 1.3074409851526922, 1.9180129853989993, 0.08330730717885483, 0.6099963275323771, -2.701356075639943, -2.0250885865242063, 3.373995914189927, -2.861008860719389, -4.867372391613269, 5.967175323821315, -2.5368395601636062, 5.466383256539465]], [[-6.112430355901956, 7.237392244488562, -1.940382318588827, -1.2123755803356147, 3.82766159818386, 3.84377809456636, -2.962120149321778, 2.2499792086403647, 2.362218896259488, -1.0520160112612413, -4.375442263823277, 1.8977165276165422, 0.33819968589359006, 2.3200979321278683, 1.0498305087194222, -1.1355740180895533], [-2.890176554004828, -0.3502642137632747, -6.042916052693385, 6.491126555783206, -2.0435071632038464, -3.4387877173579526, 4.895672712689157, -3.1677629830106078, 3.351819266120681, 1.3310861398918044, 7.979951055651244, 2.08084893521406, 1.0824722616896587, -2.793678565192715, -2.430449309575294, -0.7148942931422864]]], [[[7.691453224679167, 3.2262540933938215, -3.900905189393367, -2.0900877357304903, 0.3304839603298751, -3.7534213596722865, -1.000883473214652, -0.09862217382360075, -1.7056287293101826, -4.991006935496033, 2.736304996256376, 0.9436466601529684, -4.844764071494215, 0.3710233473219312, 3.179436410772682, 4.156816421830951], [6.151474394747245, 6.426980014728441, 4.434138633507908, -1.6917808015374411, -4.688518258721759, -2.457944779031723, 0.7642203783757021, -2.802404183872688, 6.603228607921427, 4.5240455485933735, -1.228765485090426, 5.497065703138361, -0.37299618833343157, 2.9938169011088793, 4.292646669824271, -0.14728998054609077]], [[5.307283341535301, -2.040572395484987, -0.5527395855324742, 4.418133202152195, -2.706290818631053, 2.319824093816547, -3.4423630493793036, -2.1200471853431373, -3.277347498619564, 0.2496624831666361, 6.011322056998365, 2.690143444415496, 6.9598629417140945, 0.026096375071428594, -4.270812099445923, 2.9586218375182662], [1.2386483738719982, 2.044323315973554, 1.2715300431030103, -2.2792046148473233, 2.3904403527172917, 6.295790159079626, -0.48147691332402026, 0.12180341951493814, -2.7147930774778954, 2.3894967259877964, -2.134137514543898, -6.322250458367538, -5.92913303655684, -5.539957359754066, -0.8367406613131794, 5.281880530007526]], [[5.362520802375638, -3.6965492213076105, 9.094767069535282, 3.2857964348247406, 1.0751084787019165, -1.9118648077705795, 2.2824847443191847, -1.874337516102275, 8.391430242634465, -7.7240943799473145, 3.2952838840703738, -2.049406544199854, -1.591300655375922, -0.5086040679655264, -2.094062524989371, 0.40750727551550325], [1.4157785236176663, 4.98352093166586, -4.4585209955936085, -6.172692112438849, 2.519047609621197, 2.814079161987399, 7.813520608188939, -1.5842920347308163, 4.279347209754224, -0.1743815121916634, 1.2459177992244084, 3.779217367956796, 1.2829358986488404, 3.807132128075084, 4.951062990252578, 1.2915765606944734]], [[-0.5646791517969374, -3.347574696665628, -6.1989463505179225, 0.48971603944927755, -4.400765589570637, 0.19131760661188543, 1.0549857988114817, -5.891137384324538, 3.9152846115808666, -3.2725281872747414, -4.4848798235584875, -4.698809115525951, 2.9468037695558977, 3.1033268833789194, -5.33101738655116, 1.045456889588264], [-0.34543526713730877, 0.42876643625769, 2.0021886317912805, 3.2105584645598086, 6.528448628753789, 0.49010965304230636, 1.4088896301019567, -0.8190025541653866, 2.070691829058197, 0.05859829201201357, 1.3956325879478, -5.65003195046635, 6.935731897985627, -4.340033510822999, -5.205982172167698, 0.5556142360418161]], [[-3.0809427046911253, -3.257865300514225, 4.545160408126005, -1.197230037382965, 2.1536213534190933, 2.4278281205642496, 1.673718633459577, -2.53894096521539, -5.342243369411921, -4.223964452237509, 5.495214091043101, -5.34343950818595, -2.343177394734146, 5.9114440020133845, 3.040887009867302, 0.11236692348694834], [5.992478838124411, 0.5809400424486906, 2.394969417778098, -0.4463265382058975, 0.7543295396679986, -3.49596337765308, 0.0843622460400187, -1.2824084538093707, -0.3396908592867873, 2.4702576781846255, 9.833585678971314, -1.840440831432172, 3.745004964767355, -3.5266771555929917, 2.3570976599252735, -4.380527896676136]], [[6.106730471638725, 4.3323366949341695, -3.5787995168283873, 2.3852947998461085, 0.41642165306572393, -4.981277067688448, 8.071765499334349, -0.3419743076369372, -2.617339976411341, -8.709773568206407, 4.5185678551707245, 2.9242546431548893, -3.0767365661376664, -6.7048164001148045, 0.5970060411712967, 1.6016017542581866], [-0.11751909496700462, 3.4579213330527763, 0.3572935879786799, -1.2669716571912828, 0.3992166464742066, 3.4211722238612583, -0.05564918931449651, -1.1947564301976337, -5.919688380467726, 2.456451763073619, 4.885039386796045, -4.67943860331965, 2.895912453578139, 10.019772213557705, -4.204970108925493, -3.1198313668362236]], [[-0.45890719688010595, -0.2685677426337011, 2.9881776962305997, 3.434046997510988, 1.0846317753058994, -0.08412550451753531, -3.302561458553494, 1.858115267613552, -1.2482111679555659, -4.859604010133078, -0.18411796647737205, -1.9558183805042202, -0.8421707044975502, 2.2994107956714407, 5.791037166362779, 7.766638417111677], [-4.2095048101310555, 0.047034996697047526, -2.901604597236007, -5.248055819349133, 1.8256202511936972, 1.3358034279888367, -1.7834337548172674, -4.667659496916178, -3.892265063715446, 2.7284217051213346, -2.606909293654442, 1.4260089922420662, 0.11811952681206014, -2.5741307201851518, -3.734340857362772, 1.0712833671871467]], [[-0.490454640027994, -0.10070578202927759, 5.599572102852837, 6.649720894673107, 1.4441873673996117, -4.867978729625151, 4.224919628110193, -0.32185097841345106, 0.5055112914848029, 5.3344551747155915, -6.562283506880755, 5.611349457154819, 0.838169347264994, -7.371054238476192, -1.2698025501821941, -0.7018194485049981], [2.778179067119038, -0.3540552117935944, 3.159305308240139, -3.324211951042092, -2.9199653456111947, -1.1352089132171657, 4.233693381615443, -1.449928327023283, -2.6044352389334224, 2.4111971517283255, -0.14734783640947616, 4.3630169752217345, 4.48274337013649, -6.254011746401471, -3.143884496086935, 4.956996846494306]], [[1.9492606314058014, -2.1059017354044935, 1.7932422366187502, -4.199788538069055, -7.083391546248994, -2.234776186385961, -4.2296462684787075, 2.472266898669524, -4.363259479434632, 1.8464612023453153, -1.6501352355216934, 4.461336146794939, 6.006228395257796, 1.8535220592057449, 5.203283693291339, -7.917716135346971], [7.952733609946992, -2.555724496331928, -4.5317196820294425, -1.7915199513986269, 4.84586620074899, -0.20589307707227494, -2.963072766841272, -1.5939162181835087, -2.5993367280320108, -2.931940091867244, 6.993182961254698, 7.18442081621711, 6.368886791093569, 2.8617545882321527, -5.618314451973905, 1.7871055359433234]]]], "dtype": "float64"}, {"__nd__": [[[[-0.18934762294092145, 1.3444504601336449, -1.9069326339371855, 0.8523237629565821, 2.468901713931201, 0.7214919545845229, 0.39224597117613774, 0.32187932658055535, 0.5672775680906905, -0.28311082097965284, -0.5717790206157405, -2.2139539696918447, 0.653231855107022, -0.7360525376951439, 1.212644198517262, -0.6369741389101002], [0.3584488490098457, -0.5453790623305186, 0.17769653914750827, 0.422473439385754, -1.5672465285105808, -0.49877097865430703, 0.8836093154319793, -0.42007287393054654, 0.15306659366370243, 1.2286625164339768, -1.340299462096762, -0.20276232026011864, 2.3478905790334457, 0.6321209458076988, 1.6120101561521805, -0.4888563574060917]], [[0.3677650869677704, 1.498457035389088, -1.6846723202699365, 1.103627031693854, 0.6692002936965517, -1.1214451164558281, 1.6755851000650226, -0.2247614965126635, -0.2537516673094744, -0.4576629617359849, -0.9094427080025992, 2.8383743762792037, -1.0580301438906896, 1.0946746895472814, -1.0234581523219444, 1.0844695390362225], [0.5784890258412511, -2.562494201684839, -0.48294089276439195, 1.5595550343837354, -0.9179489464581956, -0.06252728652689353, -0.051726966856484834, 1.7243760199274987, -0.7884600994442448, 0.10051853062856976, 0.4910689123928099, -1.4646189096713134, -2.5859549772580253, 0.048354032033954414, -0.18688575008190206, -0.32880881297961567]], [[-0.4654687873401875, -1.2152207554493304, 0.25177329473235693, 0.4380981096135678, 0.8626889870920064, 0.44922463956383335, -1.0171302299146698, -1.7750620937805421, -0.2509227444574525, -0.08302839980906732, 0.9670466652676414, 1.4782636355026115, -1.8854215213428345, -1.4998448771750064, -0.8743579523242608, -1.1726985204230138], [0.6833437420058257, 0.5285046995107092, 0.8878982793493168, 1.6404444833193281, 0.7905235993991685, 1.174577547356228, 0.18350576121638124, 0.4070254992419769, -1.6241389525811185, -1.3078730725680208, -0.8041318694157347, -0.7967241205739325, -0.7395267198955765, -1.8636823174097237, 1.5768543766363468, -0.8542888569353819]], [[-1.5389164310230983, -0.30090495546991003, -0.26418091262929577, -1.0313287260060258, -0.9016266170517218, -0.22974662307362537, -0.2542773895247152, 0.359506445503976, -0.009528197053672345, -0.2706843913121151, -0.7281918102479569, -0.29024252483660873, -0.1952214135697336, -0.7932723040717845, 0.006030226095427251, 0.5631056212739131], [-0.6887994994014524, -0.16776200243277742, -1.5798741963288863, -1.03492541379893, 0.25671401919613857, -0.07324765581739244, -2.165760043806217, 0.1521253212257443, -0.653258749329921, 0.02028616968098354, 0.5302670289434366, 0.9835352022187809, 1.7242993749026136, -0.9121342764488183, 0.10424041621514643, -0.6240934618322549]], [[0.3995472659332548, 0.2534269270178821, -0.5839261478831004, 1.2809011045772696, 1.5003579062340482, -1.239436862068583, 0.7442065951043049, -0.3462108035069118, 1.6519785552465454, 0.8877227412947395, -0.153961646753905, -0.3926997208769227, 0.8801160965078941, -0.24333654528717202, -0.7979354130128059, 0.4193684289076831], [-0.23869098046769377, 1.8753005507915226, -0.6900512124445851, 0.8829739933091777, 1.0586638669161936, 2.302335518837304, -1.1652960713361507, -1.1677856595624967, -0.7344479843013383, -0.5498661638933169, 0.8909875899655415, -0.30755854659095294, -0.819355676406639, -1.6821439683662904, 0.4954396357492061, 0.8863337697415065]], [[1.4663588802498502, 0.8962145083685281, -1.155509287168604, -1.1557117472211569, -0.38049977437206006, -1.0971786771076164, -1.380823660284462, 0.032281560519691305, 0.6271802535913292, -0.037272461611426375, -0.34991214366131035, 0.15100598018725808, -2.2751032605419277, -0.046765089243227774, 0.6690235536313074, 0.09576229724811373], [-1.0616937518130423, -0.6280585425667933, -0.14588031339408566, 0.5482027953811593, 0.3955698146009612, 0.1366488226400167, 0.3166101596396416, -1.4069844758780758, -1.6431210095616566, -0.4649940716348701, -0.1847935976889637, -0.3908918188034422, 1.0734088423000352, 1.978051108417339, -0.6937258036223682, -0.7342165750460977]], [[-1.188556148096582, 0.4265797235781931, 1.8948868578376603, 0.40091860722763534, -0.9518888359057226, -0.9998518717571215, 0.10174770861550034, -0.5422711842268378, -0.7953694579641029, 1.0999933574365692, 0.11289641101695624, 1.3247647756858478, -1.1631283620702522, -2.2655781091083718, -0.15677816437499625, 0.46999109363952446], [-0.13790475256021636, 0.17933033627442632, -0.06449679542032381, -0.6188223107999411, 1.0618377947118072, -0.0400887602541157, 2.358138838152814, -0.5535651903365912, -0.9354631688041514, -0.10530012227723282, 2.5070076908674244, -1.5251735726096514, -1.699779806559262, -1.9709608548725572, 0.21619909202211607, -1.1075438167344862]], [[0.4955384197802793, -0.25174092965924744, -3.3132325502374007, -1.376272165126618, -0.6124097524415884, 0.5811542242867072, 0.5398483845972162, 0.2688536916149953, -1.4755759410542257, -0.17599072664759893, -0.21305282969822015, -0.4903173772086555, -0.6600759684920898, -0.49261553853199946, -0.7221070211210193, -0.4726640905704089], [0.026289701048864845, 0.5959119052381102, -0.37249768251121856, -0.011564582016942989, -0.30280001149810404, -1.0468059870505437, 1.9011318272057725, -0.4058150917939453, 0.32110341999613556, 0.6569267365220364, 1.7701708527357394, -0.07718045787669064, 1.7021914459169292, -0.5421165656910651, -0.8559599049783363, -0.23369810823706946]], [[-0.7821360404077836, 0.4625468345088548, -1.4841120201049693, -0.13513332582237234, -1.6308231873853078, 0.8501995184877695, -0.14916275384763064, -0.729836765202496, -0.5814853670513305, 1.0860989768445741, 1.3441822622832582, 0.029356748705399956, 0.9239204620551793, 0.9022421702023155, -0.04108604718741984, 0.39978071058090414], [-0.39366941545734113, 0.33756778002681115, 1.5479795006201507, -0.2068085575345582, -0.5895092463509175, -1.606091918084821, -0.05849694148233313, -0.7776558712141016, -0.1365691116474856, 1.2300302046146183, -0.5502736958538469, 0.721358484986376, 0.8204223673926015, -0.4662545726776966, -0.9036936692218548, 0.49190529317969]]], [[[-0.7748424376350714, 2.0151553797617443, 2.820725268361806, 0.40261294619929616, -0.6634739197577162, 1.6748303014468253, -2.0251958762340436, 0.5685506017676709, -1.0342015281220265, 1.6729427611641605, -0.41890823277224204, -0.3765100233682681, 1.0059355143097122, 0.479059535215491, 0.20605401445942934, -0.21523043819031434], [-0.9507570343373676, 0.5754968214583764, 1.2048672427712057, 0.5242442678378082, -0.6046587650797319, 2.540414908555616, -1.0468942087593833, 0.04479287212590644, 0.55839572120917, -0.9326638913687012, -0.4949438624668802, -0.15680598860684566, -0.6528963897991027, 1.1490245370871972, -1.394542594908365, -0.06227679406460628]], [[1.0119722043462005, 1.0170812475492201, 0.6646205005669651, 0.018454734309266835, 0.2717194135549448, 0.20884756542690353, 0.3696639544017742, 0.7117107767755628, 0.4144518732548406, -2.2020109199324076, -0.5596891058582747, -0.5903274931987773, 0.45671921572866025, 0.012885716066347558, -1.5158920469470687, -1.073575530396053], [-0.7697002346632134, 0.8194931921759319, -0.9657258524117218, 0.46320296769714137, 1.6813570051188038, 0.9100518068806558, 0.7797137949923587, -1.1567524292128024, 0.5781155745541681, -0.5599664952340825, -1.5423149904822788, -0.5986661301701173, 0.2353827008096673, 1.150378972057663, -0.010975914569676193, 2.264336084150075]], [[-1.9345636563457578, 0.3613172380742855, -0.01868291001992957, -0.1589576410618314, 0.16126240701598776, -1.2904151829257, 1.370347399727824, 0.2897050754544625, -2.1108864163288006, 1.9739533980528148, 0.62591130283454, 1.5553272743800681, -0.8860199635583877, 0.8947252598884314, -0.40685027060830525, 1.4723727031780351], [-0.8972035603283877, -0.4036211144727192, -0.2686695990723367, 0.6218689781154865, -1.1579650733291045, 1.6590518503748408, 1.640062114816605, -0.43414256733299345, -0.4998777976345584, 0.2680333830442713, 0.0859418650162605, 1.8549041201247032, 0.5004066188270184, -1.060434353413983, 0.8010175946473618, -0.2197852648714731]], [[-1.6441098541936048, 1.5037017981970147, 0.3753637399465873, 0.42920963359041053, 0.480337004758218, 0.6509855138906528, -0.2885812977766358, 1.6432207056956563, -1.13358670807825, 1.349679588478426, 0.9895503797066358, 1.2179767121426655, -0.1323393519325693, 0.667725628184878, -0.9526681323011368, -0.7641021994406367], [-1.1378815987579072, -0.10702037094369846, -0.5418926442957446, -0.24495065742821492, 0.3169702619344202, -0.7646570057142952, 2.1521312154660657, 2.2735704191656345, 1.5156356298489597, -1.7427499078633806, 0.8542154760701247, 0.8973439595085666, 1.1023647966311032, 0.4726281051598968, 0.8047624864197406, -0.7084823884791652]], [[-0.002666129212860031, 0.05295321545456736, 1.031037787591329, -0.3198002145503273, 0.5989651804473899, 0.7683819908659891, -0.4281820049245751, -0.01659483206409203, 0.12469678250114662, -2.29402220825602, 0.6663834340666767, 1.1550911727650466, -0.052394265526388595, 0.707421063493004, -0.8077400419679064, -0.12837883496230415], [-1.7588363621644496, 0.7751119722263171, -1.4263983884817384, -1.992169652156868, -0.6756710854083023, 0.8100488164446644, -0.7323827381867019, 0.16409903983676605, 0.6487818297483998, -0.38733248916680874, -1.221818580618411, 0.9409596054992272, -1.5145159394228782, -0.19242037578331075, 0.22254712959594922, 1.3486144144851537]], [[0.30490551073093825, -1.612389343628836, 1.3976608747636294, 0.25327236868258024, -0.5333369296761634, 0.09875602600547732, 0.22228944311930182, 0.2878117640520359, 0.4198977740061699, 0.4320268028089106, 0.7302906951580344, 0.11828502116919035, 0.6398512618704889, -0.8631832454959583, -0.32185487173969557, -0.4310253690506487], [1.0947728639055723, 1.3183506567006187, 0.7566179888454229, -0.9134635174548589, -1.4967193854935399, 0.645033139222168, 0.14150921589157764, 1.1261779948895883, -1.290053211969055, 1.2134835860146411, 0.3567053130993459, 0.2179432202279926, 0.5034174158600154, -0.8749306878840502, -2.021745057801988, 0.8887898403633139]], [[-0.889088654061109, -0.04151333963844451, -0.9035161682919755, 0.23625127316200056, -0.35668578270774765, 0.8757487277005082, 0.6012517263952364, 0.5431986191141253, -0.45506607922506126, 0.4066378382910698, 1.0943549145886315, 0.9875047935321533, -1.4382477732645267, 0.31720087708083805, 0.40275201817622236, 0.5352803903438065], [-1.7152957921448675, 1.0905776773103724, -1.2218787727787854, 0.14662622726841243, -0.49313958796095747, -0.8655569463861591, 1.7691833656516114, 0.18631207110758571, 1.5669310266753331, 1.8563393547625002, 0.7914371429289206, 0.6601393806670871, -0.00456732764146071, -1.3256552483249864, 0.25858335902408286, -0.990080758831419]], [[-0.11199180245616819, -0.1653935047739032, -1.0476535213665474, 0.6839894806341457, 0.10920530400561465, -1.0095790517387022, -1.0480663192647461, -0.2801555362396602, 0.5280985859378801, -0.9444921506639953, 0.16992432551494918, -0.3879747393444699, 0.5595565893726472, 0.8142689480518304, 1.976485855339146, -0.6317216143110134], [0.6768418024455913, 0.8272843447267142, 0.6368326263198021, -1.3766966763008721, -1.0344770993014072, 1.2355937257906195, -0.134521715937077, 0.7904782688455505, -0.8658350976509096, 0.787322748537159, -0.39302712744968543, -1.0488888515677346, 1.369340867124231, -0.5267692066665984, 0.13377843052804314, 0.748990435229056]], [[1.1042025841902923, 0.6564297702155336, -0.5415061658191156, -2.1308938482374122, 1.6111078878106169, 0.6978995031127757, -0.897219701444811, -0.4735087564089369, 0.8634626966022542, -0.24173328705488212, 0.18372194457805097, 0.8189624173297644, -0.8740849855919894, -1.1130980943541695, 0.6682975307786669, 0.4038925167292101], [-0.7282126904223352, 0.6916184384805468, -1.2135903614988912, -0.6273811996567904, 0.5422889204531687, 0.6193193195960027, -1.1526660004203777, 0.02555652396213335, -3.7592763555063304, 0.7861883467613512, 0.6063377014856864, -0.3084879853668955, 0.4307996827932963, -1.073979272020325, -1.0436917608655623, -0.0815292258883002]]]], "dtype": "float64"}, {"__nd__": [9, 1], "dtype": "int64"}], "e": {"__nd__": [[[0.3708578944206238, 1.300134539604187, -1.5061577558517456, 1.1308037042617798, 0.7988824248313904, -1.1398272514343262, 1.5261963605880737, -0.2443237155675888, 0.047203704714775085, -0.2428680658340454, -0.7886300086975098, 2.324881076812744, -0.7513465285301208, 0.87872713804245, -0.9865478873252869, 0.9783490300178528], [-1.5389151573181152, -0.3009037375450134, -0.26418185234069824, -1.031327247619629, -0.9016256332397461, -0.22974714636802673, -0.2542761266231537, 0.3595060110092163, -0.0095283854752779, -0.27068445086479187, -0.7281917929649353, -0.29024046659469604, -0.1952219307422638, -0.793271005153656, 0.006029545329511166, 0.5631059408187866], [-0.3936694264411926, 0.3375677764415741, 1.547979474067688, -0.20680855214595795, -0.5895092487335205, -1.6060919761657715, -0.05849694088101387, -0.7776558995246887, -0.13656911253929138, 1.2300301790237427, -0.5502737164497375, 0.7213584780693054, 0.820422351360321, -0.4662545621395111, -0.903693675994873, 0.4919053018093109], [-1.061693549156189, -0.6280584335327148, -0.14588019251823425, 0.5482029318809509, 0.3955698311328888, 0.13664893805980682, 0.3166101276874542, -1.4069842100143433, -1.6431210041046143, -0.4649941623210907, -0.18479366600513458, -0.3908918797969818, 1.073408603668213, 1.9780505895614624, -0.6937255859375, -0.734216570854187]], [[-0.7748424410820007, 2.01515531539917, 2.820725202560425, 0.40261295437812805, -0.663473904132843, 1.6748303174972534, -2.025195837020874, 0.5685505867004395, -1.0342015027999878, 1.6729427576065063, -0.4189082384109497, -0.37651002407073975, 1.005935549736023, 0.4790595471858978, 0.20605401694774628, -0.21523043513298035], [-0.7748424410820007, 2.01515531539917, 2.820725202560425, 0.40261295437812805, -0.663473904132843, 1.6748303174972534, -2.025195837020874, 0.5685505867004395, -1.0342015027999878, 1.6729427576065063, -0.4189082384109497, -0.37651002407073975, 1.005935549736023, 0.4790595471858978, 0.20605401694774628, -0.21523043513298035], [-0.9507570266723633, 0.5754967927932739, 1.2048672437667847, 0.5242442488670349, -0.6046587824821472, 2.540414810180664, -1.0468941926956177, 0.04479287192225456, 0.5583957433700562, -0.9326639175415039, -0.49494385719299316, -0.15680599212646484, -0.6528964042663574, 1.149024486541748, -1.3945425748825073, -0.062276795506477356], [-0.9507570266723633, 0.5754967927932739, 1.2048672437667847, 0.5242442488670349, -0.6046587824821472, 2.540414810180664, -1.0468941926956177, 0.04479287192225456, 0.5583957433700562, -0.9326639175415039, -0.49494385719299316, -0.15680599212646484, -0.6528964042663574, 1.149024486541748, -1.3945425748825073, -0.062276795506477356]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[-0.3904497170033783, 0.39584135204924314], [0.1926794518754327, 0.1676606309601963]], [[0.34788592674123575, -0.6023802982258547], [1.1427930120917758, 0.27354639952849363]], [[0.7830752422159135, -0.22525795347144334], [-1.3516848914969317, -0.48802329001360895]], [[-1.1276069462163583, -0.3712511418693076], [-0.07327913788046607, -0.5236553267300357]]], "dtype": "float64"}, {"__nd__": [[[[0.22573948918846445, 1.5873218323665148], [-1.6873490349120028, 0.9608910016229573]], [[-0.07729065483441719, -0.23981477169885107], [1.92877062739379, -0.37094212831199685]], [[-0.4339747371936108, -0.6976306747732555], [-2.471994913410796, 0.20476581974490835]]], [[[-0.4311734328861134, -0.2773828351249614], [1.6746335682413662, 1.6774191794881081]], [[-0.8669856825312358, 0.8496238084427864], [0.2312302356420075, -1.0069154149849893]], [[1.0685687234708146, 0.12408464067230242], [-0.7633863839875348, -0.4773531910042446]]], [[[0.5901365722848303, -1.0923988473450827], [-2.1412682008262145, -1.4483377321503053]], [[-0.5280459655795713, -0.6188875607420169], [0.09744785379909937, -1.331270678691157]], [[-1.5916117002647283, 1.1353060182874406], [-0.45672208435632444, -0.7263496088103023]]], [[[-0.3331801860562357, -1.5432381853551487], [1.1173494741528582, -0.5501217455822588]], [[-0.6453666071478035, 0.5247144408965728], [-2.209284650096351, 1.1737900555501088]], [[-1.2160382295459589, 1.26939269314075], [-0.6288913445132233, 1.4155434496613535]]]], "dtype": "float64"}, {"__nd__": [[[[-0.8981544385351259, 0.5279146441946528], [-0.4333837500760284, 1.3940680485259649]], [[0.9495831269642885, -0.3797101448592878], [-0.7991719203807193, 0.24909424763794613]], [[1.2919271274939632, 1.6453497655390539], [-0.11140816752226937, 0.9315344485244952]]], [[[-1.0300223865632292, 1.0187927948136453], [0.15080954843381617, -0.8631773874262183]], [[-0.9346055494777593, 0.5022889907150279], [0.3229598247454781, 0.21354569329035453]], [[-0.40210336209399467, 0.7991037043880559], [-0.7066215928782348, 1.449707868395774]]], [[[-0.12478347109058295, -1.5833529105529547], [-0.536991869819764, 0.09946000702845453]], [[-0.022042772901217037, 1.3613855135912616], [-0.7657251133996597, -0.5112245030741289]], [[-1.223440621080536, 1.1526399310892301], [0.3506082644918488, 0.45101060160839773]]], [[[-0.47838420199609827, 0.9100439400500212], [-0.6258364358001395, -0.8950177878616552]], [[1.5609514301721585, -0.08488957895020033], [0.524007233574319, -0.8459749890453009]], [[-0.5601294803917747, -0.36386559903637106], [0.06746239933266905, 0.3849529060058524]]]], "dtype": "float64"}, {"__nd__": [3, 3, 2, 1], "dtype": "int64"}], "e": {"__nd__": [[[0.23573318123817444, 0.578461766242981], [-0.5100226402282715, 0.7789279818534851]], [[-0.7350783348083496, 0.8188188076019287], [0.11408588290214539, -0.5404993891716003]], [[-0.09056925773620605, -0.6027103066444397], [-0.5602195858955383, 0.03744559362530708]], [[-0.47838419675827026, 0.9100439548492432], [-0.6258364319801331, -0.8950178027153015]]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(gqa_decode, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")